In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
import warnings

# ========== PARAMETER & KONFIGURASI ==========

INPUT_FILENAME = 'FINAL_Merged_SmartMeter_dan_Cuaca.csv'
OUTPUT_XGB_TRAIN = 'data_xgb_train.csv'
OUTPUT_XGB_TEST = 'data_xgb_test.csv'
OUTPUT_LSTM_TRAIN = 'data_lstm_train.csv'
OUTPUT_LSTM_TEST = 'data_lstm_test.csv'

# Kolom target prediksi dan kolom kategorikal yang ingin di-One Hot Encoding
TARGET_COLUMN = 'Konsumsi Energi'
CATEGORICAL_FEATURES = ['Weather Description']
# Kolom yang tidak relevan untuk modelling
IRRELEVANT_COLS = [
    'id_id', 'id_n_id', 'id_meter_id', 'id_create_date', 'updated_at',
    'created_at', 'resample_date', 'data_points_count', 'lokasi', 'rssi'
]
# Nama kolom waktu
TIME_COLUMN = 'id_time'
# Ratio train/test split (70%:30%)
TRAIN_RATIO = 0.7

# ========== FUNGSIONALITAS ==========

def prepare_data_for_modeling(
    input_filename=INPUT_FILENAME,
    target_column=TARGET_COLUMN,
    categorical_features=CATEGORICAL_FEATURES,
    irrelevant_cols=IRRELEVANT_COLS,
    time_column=TIME_COLUMN,
    train_ratio=TRAIN_RATIO
):
    warnings.filterwarnings('ignore', category=pd.errors.SettingWithCopyWarning)
    warnings.filterwarnings('ignore', category=FutureWarning)

    print(f"Memuat data dari file: {input_filename}")
    try:
        df = pd.read_csv(input_filename)
    except FileNotFoundError:
        print(f"Error: File '{input_filename}' tidak ditemukan.")
        return

    before_rows = len(df)
    df.dropna(how='all', inplace=True)
    dropped_fully_na = before_rows - len(df)
    if dropped_fully_na:
        print(f"Drop {dropped_fully_na} baris kosong (semua kolom NaN).")

    # Rename kolom agar konsisten
    if 'is_day ()' in df.columns:
        df.rename(columns={'is_day ()':'is_day'}, inplace=True)

    # Drop baris yang targetnya NaN
    before_rows = len(df)
    df.dropna(subset=[target_column], inplace=True)
    print(f"Drop {before_rows-len(df)} baris yang targetnya NaN.")

    # Drop kolom tak relevan
    df.drop(columns=[col for col in irrelevant_cols if col in df.columns], inplace=True, errors='ignore')

    # Konversi kolom waktu dan urutkan
    df[time_column] = pd.to_datetime(df[time_column])
    df.sort_values(time_column, inplace=True)
    df.set_index(time_column, inplace=True)

    # ==== SPLIT TRAIN/TEST (KRONOLOGIS) ====
    split_idx = int(len(df) * train_ratio)
    df_train = df.iloc[:split_idx].copy()
    df_test = df.iloc[split_idx:].copy()
    print(f"Train Size: {len(df_train)} | Test Size: {len(df_test)}")

    # ==== One-Hot Encoding FITUR TRAIN + APPLY KE TEST ====
    train_cat = pd.get_dummies(df_train, columns=categorical_features, drop_first=True)
    test_cat = pd.get_dummies(df_test, columns=categorical_features, drop_first=True)

    # Pastikan kolom OHE sama (kadang kategori langka hanya muncul di test)
    train_cat, test_cat = train_cat.align(test_cat, join='left', axis=1, fill_value=0)

    # ==== Interpolasi FITUR (hindari target column) ====
    feature_cols = [col for col in train_cat.columns if col != target_column]
    train_cat[feature_cols] = train_cat[feature_cols].interpolate(method='time', axis=0)
    test_cat[feature_cols] = test_cat[feature_cols].interpolate(method='time', axis=0)

    # ==== Konversi ke numerik semua (ada kemungkinan OHE menghasilkan objek) ====
    train_cat = train_cat.apply(pd.to_numeric, errors='coerce')
    test_cat = test_cat.apply(pd.to_numeric, errors='coerce')

    # ==== Drop NaN pasca-interpolasi/konversi, & log jumlahnya ====
    before_drop_train = len(train_cat)
    before_drop_test = len(test_cat)
    train_cat.dropna(inplace=True)
    test_cat.dropna(inplace=True)
    print(f"Drop {before_drop_train-len(train_cat)} baris di train, {before_drop_test-len(test_cat)} baris di test setelah interpolasi dan konversi type.")

    # ==== FEATURE ENGINEERING XGBOOST (TRAIN & TEST) ====
    def feature_engineering_xgb(df_):
        df = df_.copy()
        df['hour'] = df.index.hour
        df['dayofweek'] = df.index.dayofweek
        df['month'] = df.index.month
        df['dayofyear'] = df.index.dayofyear
        df['quarter'] = df.index.quarter
        df['weekofyear'] = df.index.isocalendar().week.astype(int)
        return df

    xgb_train = feature_engineering_xgb(train_cat)
    xgb_test = feature_engineering_xgb(test_cat)

    xgb_train.to_csv(OUTPUT_XGB_TRAIN)
    xgb_test.to_csv(OUTPUT_XGB_TEST)
    print(f"Data untuk XGBoost disimpan ke {OUTPUT_XGB_TRAIN} dan {OUTPUT_XGB_TEST}. Fitur: {len(xgb_train.columns)}")

    # ==== SCALING UNTUK LSTM (TRAIN FIT, TEST TRANSFORM) ====
    lstm_feature_cols = [col for col in train_cat.columns]  # scaling semua, termasuk target
    scaler = MinMaxScaler()
    # Fit hanya pada train, lalu transform dua set
    train_lstm_scaled = pd.DataFrame(
        scaler.fit_transform(train_cat[lstm_feature_cols]), 
        columns=lstm_feature_cols, 
        index=train_cat.index
    )
    test_lstm_scaled = pd.DataFrame(
        scaler.transform(test_cat[lstm_feature_cols]), 
        columns=lstm_feature_cols, 
        index=test_cat.index
    )

    train_lstm_scaled.to_csv(OUTPUT_LSTM_TRAIN)
    test_lstm_scaled.to_csv(OUTPUT_LSTM_TEST)
    print(f"Data untuk LSTM (scaled) disimpan ke {OUTPUT_LSTM_TRAIN} dan {OUTPUT_LSTM_TEST}")

    print("\nContoh data XGBoost (train):")
    print(xgb_train.head())
    print("\nContoh data LSTM (train, scaled):")
    print(train_lstm_scaled.head())
    print("\nPipeline selesai.")

if __name__ == "__main__":
    prepare_data_for_modeling()

Memuat data dari file: FINAL_Merged_SmartMeter_dan_Cuaca.csv
Drop 656 baris kosong (semua kolom NaN).
Drop 29 baris yang targetnya NaN.
Train Size: 6731 | Test Size: 2885
Drop 0 baris di train, 0 baris di test setelah interpolasi dan konversi type.
Data untuk XGBoost disimpan ke data_xgb_train.csv dan data_xgb_test.csv. Fitur: 48
Data untuk LSTM (scaled) disimpan ke data_lstm_train.csv dan data_lstm_test.csv

Contoh data XGBoost (train):
            id_stand_energy_kirim       id_v1       id_v2       id_v3  \
id_time                                                                 
2024-01-02           8.990658e+08  225.269663  226.303371  227.977528   
2024-01-02           2.678070e+06  226.469565  229.000000  226.982609   
2024-01-02           8.485404e+07  226.682927  227.487805  229.146342   
2024-01-02           1.924114e+07  226.781513  227.016807  228.873950   
2024-01-02           6.298761e+07  226.318965  226.879310  228.482759   

                id_i1      id_i2      id_i3  i

In [2]:
import os
import argparse
import random

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, classification_report, mean_squared_error, r2_score)

from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping

import sys

# ----------- CONFIGURABLE PARAMETERS -----------
DEFAULT_DATA_PATH = 'FINAL_Merged_SmartMeter_dan_Cuaca.csv'
DEFAULT_MODEL_TYPE = 'rfr'  # 'rfc', 'rfr', 'keras_mlp_clf', 'keras_mlp_reg'
DEFAULT_MODEL_SAVE_PATH = 'praproses/model_output_harian'
DEFAULT_TARGET_COLUMN = 'Konsumsi Energi'
DEFAULT_USE_GPU = True  # Set False untuk force CPU

# ============= GPU CONFIGURATION =============
def configure_gpu(use_gpu=True):
    """Configure GPU/CPU usage for TensorFlow."""
    print("\n" + "="*50)
    print("GPU CONFIGURATION")
    print("="*50)
    
    # Check available GPUs
    gpus = tf.config.list_physical_devices('GPU')
    
    if gpus:
        print(f"✓ Found {len(gpus)} GPU(s):")
        for i, gpu in enumerate(gpus):
            print(f"  - GPU {i}: {gpu.name}")
        
        if use_gpu:
            try:
                # Enable memory growth (prevents TF from allocating all GPU memory)
                for gpu in gpus:
                    tf.config.experimental.set_memory_growth(gpu, True)
                
                # Optional: Set specific GPU if multiple available
                # tf.config.set_visible_devices(gpus[0], 'GPU')
                
                print("✓ GPU training ENABLED")
                print("✓ Memory growth enabled (dynamic allocation)")
            except RuntimeError as e:
                print(f"✗ GPU configuration error: {e}")
        else:
            # Force CPU usage
            tf.config.set_visible_devices([], 'GPU')
            print("⚠ GPU training DISABLED (forced CPU mode)")
    else:
        print("✗ No GPU detected. Training will use CPU.")
        print("  To enable GPU:")
        print("  1. Install CUDA Toolkit & cuDNN")
        print("  2. Install: pip install tensorflow-gpu")
    
    # Show final device placement
    print(f"\nTensorFlow version: {tf.__version__}")
    print(f"GPU Available: {tf.test.is_gpu_available()}")
    print(f"Built with CUDA: {tf.test.is_built_with_cuda()}")
    print("="*50 + "\n")

def set_all_seeds(seed):
    """Ensure reproducibility across random libraries."""
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

# ---------------- DATA HANDLING ----------------

def load_dataset(path, target_column):
    """Load the dataset and split features/target."""
    print(f"\nLoading dataset from: {path}")
    df = pd.read_csv(path)
    assert target_column in df.columns, f"Target column '{target_column}' not found!"
    
    # Drop rows where target is NaN
    before_drop = len(df)
    df = df.dropna(subset=[target_column])
    after_drop = len(df)
    if before_drop > after_drop:
        print(f"Dropped {before_drop - after_drop} rows with NaN in target column '{target_column}'")
    
    X = df.drop(columns=[target_column])
    y = df[target_column]
    print(f"Loaded dataset with shape {df.shape}")
    return X, y

def identify_feature_types(X):
    """Return separate lists for numerical and categorical feature names."""
    num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = X.select_dtypes(exclude=[np.number]).columns.tolist()
    return num_cols, cat_cols

def preprocess_features(X_train, X_test):
    """
    1. Handle missing values in numerical features with forward fill then backward fill
    2. Scale numerical features (fit on train only, transform both).
    3. One-hot encode categorical features (align columns for train/test).
    4. Concatenate back into unified numpy arrays.
    """
    num_cols, cat_cols = identify_feature_types(X_train)

    # Handle missing values in numerical columns before scaling
    X_train_num_df = X_train[num_cols].copy()
    X_test_num_df = X_test[num_cols].copy()
    
    # Fill NaN with forward fill, then backward fill, then mean (as last resort)
    for col in num_cols:
        X_train_num_df[col] = X_train_num_df[col].fillna(method='ffill').fillna(method='bfill').fillna(X_train_num_df[col].mean())
        X_test_num_df[col] = X_test_num_df[col].fillna(method='ffill').fillna(method='bfill').fillna(X_train_num_df[col].mean())

    # 1. Scaling numerical features
    scaler = StandardScaler()
    X_train_num = scaler.fit_transform(X_train_num_df)
    X_test_num = scaler.transform(X_test_num_df)

    # 2. One-hot encode categorical features
    X_train_cat = pd.get_dummies(X_train[cat_cols], drop_first=True) if cat_cols else pd.DataFrame(index=X_train.index)
    X_test_cat = pd.get_dummies(X_test[cat_cols], drop_first=True) if cat_cols else pd.DataFrame(index=X_test.index)

    # 3. Align dummy columns between train/test
    X_train_cat, X_test_cat = X_train_cat.align(X_test_cat, join='left', axis=1, fill_value=0)

    # 4. Turn to numpy for concat (handle empty DataFrames if no categorical features)
    if not X_train_cat.empty:
        X_train_all = np.concatenate([X_train_num, X_train_cat.values], axis=1)
        X_test_all = np.concatenate([X_test_num, X_test_cat.values], axis=1)
    else:
        X_train_all = X_train_num
        X_test_all = X_test_num

    return X_train_all, X_test_all, scaler

def encode_labels(y_train, y_test, is_classification):
    """Only encode class labels for classification tasks; leave regression labels unchanged."""
    if is_classification and y_train.dtype == 'O':
        le = LabelEncoder()
        y_train_enc = le.fit_transform(y_train)
        y_test_enc = le.transform(y_test)
        return y_train_enc, y_test_enc, le
    else:
        # Convert to numpy arrays for consistency; regression or already numeric
        return np.array(y_train, dtype=np.float64), np.array(y_test, dtype=np.float64), None

def clean_nan_from_arrays(X_train, X_test, y_train, y_test):
    """Remove rows with NaN from train and test sets - robust version."""
    
    # Ensure y is numeric and 1D
    y_train = np.asarray(y_train, dtype=np.float64).ravel()
    y_test = np.asarray(y_test, dtype=np.float64).ravel()
    
    # Clean train set
    X_train_nan_mask = np.isnan(X_train).any(axis=1) if X_train.dtype.kind in 'fc' else np.zeros(len(X_train), dtype=bool)
    y_train_nan_mask = np.isnan(y_train)
    train_valid_mask = ~(X_train_nan_mask | y_train_nan_mask)
    
    X_train_clean = X_train[train_valid_mask]
    y_train_clean = y_train[train_valid_mask]
    train_dropped = len(X_train) - len(X_train_clean)
    if train_dropped > 0:
        print(f"Dropped {train_dropped} rows with NaN from training set")
    
    # Clean test set
    X_test_nan_mask = np.isnan(X_test).any(axis=1) if X_test.dtype.kind in 'fc' else np.zeros(len(X_test), dtype=bool)
    y_test_nan_mask = np.isnan(y_test)
    test_valid_mask = ~(X_test_nan_mask | y_test_nan_mask)
    
    X_test_clean = X_test[test_valid_mask]
    y_test_clean = y_test[test_valid_mask]
    test_dropped = len(X_test) - len(X_test_clean)
    if test_dropped > 0:
        print(f"Dropped {test_dropped} rows with NaN from test set")
    
    return X_train_clean, X_test_clean, y_train_clean, y_test_clean

# ---------------- MODEL UTILITIES --------------

def build_model(model_type, input_dim=None, hyperparams=None):
    """Build and return the appropriate model."""
    if hyperparams is None:
        hyperparams = {}
    if model_type == 'rfc':
        print("⚠ Note: Random Forest Classifier uses CPU only (scikit-learn)")
        return RandomForestClassifier(**hyperparams)
    elif model_type == 'rfr':
        print("⚠ Note: Random Forest Regressor uses CPU only (scikit-learn)")
        return RandomForestRegressor(**hyperparams)
    elif model_type == 'keras_mlp_clf':  # Keras dense MLP classifier
        print("✓ Building Keras MLP Classifier (GPU-compatible)")
        n_classes = hyperparams.get('n_classes', 2)
        model = Sequential([
            Dense(128, activation='relu', input_shape=(input_dim,)),
            Dense(64, activation='relu'),
            Dense(32, activation='relu'),
            Dense(n_classes, activation='softmax' if n_classes > 2 else 'sigmoid')
        ])
        model.compile(
            optimizer='adam',
            loss='sparse_categorical_crossentropy' if n_classes > 2 else 'binary_crossentropy',
            metrics=['accuracy'])
        return model
    elif model_type == 'keras_mlp_reg':
        print("✓ Building Keras MLP Regressor (GPU-compatible)")
        model = Sequential([
            Dense(128, activation='relu', input_shape=(input_dim,)),
            Dense(64, activation='relu'),
            Dense(32, activation='relu'),
            Dense(1, activation='linear')
        ])
        model.compile(optimizer='adam', loss='mse', metrics=['mse', 'mae'])
        return model
    else:
        raise ValueError(f"Unsupported model type: {model_type}")

def save_model(model, path, model_type):
    """Save model (scikit-learn or keras)."""
    if 'keras' in model_type:
        model.save(path)
    else:
        import joblib
        joblib.dump(model, path)
    print(f"Model saved to: {path}")

def evaluate_model(model, X_test, y_test, model_type, classification=True):
    """Predict and report relevant metrics for the test set."""
    if 'keras' in model_type:
        y_pred = model.predict(X_test)
        if classification:
            if y_pred.ndim > 1 and y_pred.shape[1] > 1:  # multiclass
                y_pred_labels = np.argmax(y_pred, axis=1)
            else:  # binary
                y_pred_labels = (y_pred > 0.5).astype(int).reshape(-1)
        else:
            y_pred_labels = y_pred.ravel()
    else:
        y_pred_labels = model.predict(X_test)
        if classification:
            pass  # For sklearn classifiers, y_pred_labels is already class predictions

    results = {}
    if classification:
        results['accuracy'] = accuracy_score(y_test, y_pred_labels)
        results['precision'] = precision_score(y_test, y_pred_labels, average='weighted', zero_division=0)
        results['recall'] = recall_score(y_test, y_pred_labels, average='weighted', zero_division=0)
        results['f1'] = f1_score(y_test, y_pred_labels, average='weighted', zero_division=0)
        print("\nClassification Report:")
        print(classification_report(y_test, y_pred_labels, zero_division=0))
    else:
        results['rmse'] = np.sqrt(mean_squared_error(y_test, y_pred_labels))
        results['r2'] = r2_score(y_test, y_pred_labels)
        results['mae'] = np.mean(np.abs(y_test - y_pred_labels))
        print(f"\nRegression Metrics:")
        print(f"  RMSE: {results['rmse']:.4f}")
        print(f"  MAE:  {results['mae']:.4f}")
        print(f"  R² Score: {results['r2']:.4f}")

    print("\nEvaluation metrics:", results)
    return results

# ------------------- MAIN ----------------------

def main(
    data_path=DEFAULT_DATA_PATH,
    model_type=DEFAULT_MODEL_TYPE,
    model_save_path=DEFAULT_MODEL_SAVE_PATH,
    target_column=DEFAULT_TARGET_COLUMN,
    test_ratio=0.2,
    random_seed=42,
    use_gpu=DEFAULT_USE_GPU,
    hyperparams=None
):
    if hyperparams is None:
        hyperparams = {}

    # Configure GPU before any TensorFlow operations
    configure_gpu(use_gpu=use_gpu)
    
    set_all_seeds(random_seed)

    # 1. LOAD DATASET (drops NaN in target)
    X, y = load_dataset(data_path, target_column)

    # 2. DETERMINE TASK TYPE
    is_classification = (model_type in ['rfc', 'keras_mlp_clf'])

    # 3. SPLIT TRAIN/TEST
    stratify = y if is_classification and len(np.unique(y)) > 1 else None
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_ratio, random_state=random_seed, stratify=stratify
    )
    print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")

    # 4. FEATURE PREPROCESSING: SCALE NUMBERS, OHE CATEGORICAL
    X_train_pp, X_test_pp, scaler = preprocess_features(X_train, X_test)

    # 5. LABEL ENCODING FOR TARGET (IF NEEDED)
    y_train_enc, y_test_enc, label_encoder = encode_labels(y_train, y_test, is_classification)

    # 6. CLEAN NaN FROM PREPROCESSED DATA (if any remain)
    X_train_pp, X_test_pp, y_train_enc, y_test_enc = clean_nan_from_arrays(
        X_train_pp, X_test_pp, y_train_enc, y_test_enc
    )
    print(f"After cleaning NaN - Train: {X_train_pp.shape}, Test: {X_test_pp.shape}")

    # 7. MODEL BUILDING
    input_dim = X_train_pp.shape[1]
    if 'keras' in model_type:
        if is_classification:
            hyperparams['n_classes'] = len(np.unique(y_train_enc))
        model = build_model(model_type, input_dim=input_dim, hyperparams=hyperparams)
    else:
        model = build_model(model_type, hyperparams=hyperparams)

    # 8. MODEL TRAINING
    print(f"\nTraining model: {model_type}")
    if 'keras' in model_type:
        epochs = hyperparams.get('epochs', 50)
        batch_size = hyperparams.get('batch_size', 64)
        
        # Early stopping callback
        early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
        
        # Show which device is being used
        print(f"Training on: {'/GPU:0' if tf.config.list_physical_devices('GPU') and use_gpu else '/CPU:0'}")
        
        model.fit(
            X_train_pp, y_train_enc,
            epochs=epochs, batch_size=batch_size,
            validation_data=(X_test_pp, y_test_enc),
            callbacks=[early_stop],
            verbose=2
        )
    else:
        model.fit(X_train_pp, y_train_enc)

    # 9. SAVE TRAINED MODEL
    save_model(model, model_save_path, model_type)

    # 10. EVALUATION ON TEST
    print("\nEvaluating model on test data...")
    results = evaluate_model(
        model, X_test_pp, y_test_enc, 
        model_type, classification=is_classification
    )

    print("\n==== FINAL EVALUATION RESULTS ====")
    for k, v in results.items():
        print(f"{k.upper():10s}: {v:.4f}" if isinstance(v, float) else f"{k.upper():10s}: {v}")
    
    return results

# ----------------- ENTRY POINT -----------------

if __name__ == "__main__":
    # Deteksi apakah script dijalankan di Jupyter/IPython
    is_jupyter = 'ipykernel' in sys.argv[0] or 'IPython' in sys.modules
    
    if is_jupyter:
        # Jika di Jupyter, langsung panggil main() dengan default parameters
        print("Running in Jupyter/IPython environment with default parameters...")
        main(
            data_path=DEFAULT_DATA_PATH,
            model_type=DEFAULT_MODEL_TYPE,
            model_save_path=DEFAULT_MODEL_SAVE_PATH,
            target_column=DEFAULT_TARGET_COLUMN,
            test_ratio=0.2,
            random_seed=42,
            use_gpu=DEFAULT_USE_GPU
        )
    else:
        # Jika di command line, gunakan argparse
        parser = argparse.ArgumentParser(description="Best-practice ML pipeline with GPU support.")
        parser.add_argument('--data_path', type=str, default=DEFAULT_DATA_PATH,
                           help='Path to the input CSV file')
        parser.add_argument('--target_column', type=str, default=DEFAULT_TARGET_COLUMN,
                           help='Name of the target column')
        parser.add_argument('--model_type', type=str, default=DEFAULT_MODEL_TYPE,
                           help="'rfc', 'rfr', 'keras_mlp_clf', or 'keras_mlp_reg'")
        parser.add_argument('--model_save_path', type=str, default=DEFAULT_MODEL_SAVE_PATH,
                           help='Path to save the trained model')
        parser.add_argument('--test_ratio', type=float, default=0.2,
                           help='Test set ratio (0.0 - 1.0)')
        parser.add_argument('--random_seed', type=int, default=42,
                           help='Random seed for reproducibility')
        parser.add_argument('--use_gpu', type=bool, default=DEFAULT_USE_GPU,
                           help='Enable GPU training for Keras models')
        
        args = parser.parse_args()
        
        main(
            data_path=args.data_path,
            model_type=args.model_type,
            model_save_path=args.model_save_path,
            target_column=args.target_column,
            test_ratio=args.test_ratio,
            random_seed=args.random_seed,
            use_gpu=args.use_gpu
        )

Running in Jupyter/IPython environment with default parameters...

GPU CONFIGURATION
✓ Found 1 GPU(s):
  - GPU 0: /physical_device:GPU:0
✓ GPU training ENABLED
✓ Memory growth enabled (dynamic allocation)

TensorFlow version: 2.20.0
Instructions for updating:
Use `tf.config.list_physical_devices('GPU')` instead.
GPU Available: True
Built with CUDA: True


Loading dataset from: FINAL_Merged_SmartMeter_dan_Cuaca.csv
Dropped 685 rows with NaN in target column 'Konsumsi Energi'
Loaded dataset with shape (9616, 47)
Train shape: (7692, 46), Test shape: (1924, 46)


I0000 00:00:1763191323.539118   27550 gpu_device.cc:2020] Created device /device:GPU:0 with 2244 MB memory:  -> device: 0, name: NVIDIA GeForce GTX 1650, pci bus id: 0000:01:00.0, compute capability: 7.5
/tmp/ipykernel_27550/3569470970.py:121: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  X_train_num_df[col] = X_train_num_df[col].fillna(method='ffill').fillna(method='bfill').fillna(X_train_num_df[col].mean())
/tmp/ipykernel_27550/3569470970.py:122: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  X_test_num_df[col] = X_test_num_df[col].fillna(method='ffill').fillna(method='bfill').fillna(X_train_num_df[col].mean())


After cleaning NaN - Train: (7692, 8779), Test: (1924, 8779)
⚠ Note: Random Forest Regressor uses CPU only (scikit-learn)

Training model: rfr
Model saved to: praproses/model_output_harian

Evaluating model on test data...

Regression Metrics:
  RMSE: 326972.0426
  MAE:  114901.6951
  R² Score: 0.8868

Evaluation metrics: {'rmse': np.float64(326972.0425693856), 'r2': 0.8868354940983225, 'mae': np.float64(114901.69511879914)}

==== FINAL EVALUATION RESULTS ====
RMSE      : 326972.0426
R2        : 0.8868
MAE       : 114901.6951


In [1]:
import tensorflow as tf
import os

# Suppress TensorFlow warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# CRITICAL: Enable GPU memory growth
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        
        # Optional: Set memory limit (GTX 1650 = 4GB VRAM)
        # Limit to 3.5GB to be safe
        tf.config.set_logical_device_configuration(
            gpus[0],
            [tf.config.LogicalDeviceConfiguration(memory_limit=3584)]
        )
        
        print(f"✅ GPU memory configured for {len(gpus)} GPU(s)")
        
    except RuntimeError as e:
        print(f"GPU config error: {e}")

# Verify GPU
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")

2025-11-15 15:27:09.750877: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-15 15:27:10.308062: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-11-15 15:27:12.812775: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


✅ GPU memory configured for 1 GPU(s)
GPU Available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
import os
import argparse
import random
import logging

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report, confusion_matrix,
    mean_squared_error, r2_score, mean_absolute_error
)
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.utils.class_weight import compute_class_weight
from scipy.stats import randint

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping
import keras_tuner as kt

import sys
import json
import joblib

# ------------- LOGGING CONFIGURATION ---------------
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler("ml_pipeline_log.txt", mode='w')
    ]
)

# ------------- DEFAULT CONSTANTS ---------------
DEFAULT_DATA_PATH = 'FINAL_Merged_SmartMeter_dan_Cuaca.csv'
DEFAULT_MODEL_TYPE = 'rfr'  # 'rfc', 'rfr', 'keras_mlp_clf', 'keras_mlp_reg'
DEFAULT_MODEL_SAVE_PATH = 'praproses/model_output_harian.joblib'
DEFAULT_TARGET_COLUMN = 'Konsumsi Energi'
DEFAULT_USE_GPU = True
DEFAULT_PLOT_FILENAME = "performance_plot_actual_vs_pred.png"
DEFAULT_CONFUSION_MATRIX_FILENAME = "confusion_matrix.png"
DEFAULT_CORR_HEATMAP_FILENAME = "feature_correlation_heatmap.png"
DEFAULT_KERAS_HISTORY_FILENAME_PREFIX = "keras_training_history"
DEFAULT_RESIDUAL_PLOT_FILENAME = "regression_diagnostics.png"
DEFAULT_FEATURE_IMPORTANCE_FILENAME = "feature_importance.png"

# --- Constants for new features ---
DEFAULT_ENABLE_TUNING = True
DEFAULT_HANDLE_IMBALANCE = True
DEFAULT_TUNING_ITER = 10 
DEFAULT_CV_FOLDS = 3

# --- Batas untuk deteksi High Cardinality ---
HIGH_CARDINALITY_THRESHOLD = 50 


def configure_gpu(use_gpu=True):
    logging.info("="*50)
    logging.info("GPU CONFIGURATION")
    gpus = tf.config.list_physical_devices('GPU')
    if gpus:
        logging.info(f"✓ Found {len(gpus)} GPU(s):")
        for i, gpu in enumerate(gpus):
            logging.info(f"  - GPU {i}: {gpu.name}")
        if use_gpu:
            try:
                for gpu in gpus:
                    tf.config.experimental.set_memory_growth(gpu, True)
                logging.info("✓ GPU training ENABLED (dynamic allocation enabled)")
            # Menangkap error jika runtime sudah diinisialisasi
            except (RuntimeError, ValueError) as e:
                logging.warning(f"✗ Tidak dapat mengatur memory growth (mungkin runtime sudah diinisialisasi?): {e}")
        else:
            tf.config.set_visible_devices([], 'GPU')
            logging.info("⚠ GPU training DISABLED (forced CPU mode)")
    else:
        logging.warning("✗ No GPU detected. Using CPU.")
    logging.info(f"TensorFlow version: {tf.__version__}")
    logging.info(f"GPU Available: {tf.config.list_physical_devices('GPU') != []}")
    logging.info("="*50)

def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

def load_dataset(path, target_column):
    logging.info(f"Loading dataset from: {path}")
    try:
        df = pd.read_csv(path)
    except FileNotFoundError:
        logging.error(f"FATAL: Data file not found at {path}")
        sys.exit(1)
    
    assert target_column in df.columns, f"Target column '{target_column}' not found!"
    
    before_drop = len(df)
    df = df.dropna(subset=[target_column])
    after_drop = len(df)
    if before_drop > after_drop:
        logging.info(f"Dropped {before_drop - after_drop} rows with NaN in target column '{target_column}'")
        
    X = df.drop(columns=[target_column])
    y = df[target_column]
    logging.info(f"Loaded dataset with shape {df.shape}")
    
    # Coba konversi kolom yang terlihat seperti datetime
    for col in X.columns:
        if X[col].dtype == 'object':
            try:
                # Coba konversi, tapi jangan paksakan jika error
                pd.to_datetime(X[col], errors='raise', format=None)
                X[col] = pd.to_datetime(X[col])
                logging.info(f"  -> Kolom '{col}' terdeteksi sebagai string dan dikonversi ke datetime.")
            except (ValueError, TypeError):
                continue # Biarkan sebagai 'object' (kategori)
    
    return X, y

def identify_feature_types(X):
    """Mendeteksi tipe kolom: numerik, kategorikal (teks), dan datetime."""
    num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    # Deteksi kolom datetime secara spesifik
    dt_cols = X.select_dtypes(include=['datetime', 'datetime64']).columns.tolist()
    # Kolom kategori adalah sisanya (object/string)
    cat_cols = X.select_dtypes(exclude=[np.number, 'datetime', 'datetime64']).columns.tolist()
    
    logging.info(f"Identified {len(num_cols)} numerical cols, {len(dt_cols)} datetime cols, {len(cat_cols)} categorical cols.")
    return num_cols, cat_cols, dt_cols

def preprocess_features(X_train, X_test):
    """
    Memproses fitur:
    1. Ekstrak fitur dari datetime.
    2. Cek kardinalitas: buang kolom kategori yang high-cardinality.
    3. Imputasi dan scaling kolom numerik (termasuk yg dari datetime).
    4. One-hot encode kolom kategori yang low-cardinality.
    """
    logging.info("Starting feature preprocessing...")
    num_cols, cat_cols, dt_cols = identify_feature_types(X_train)
    
    # Salin dataframe numerik (awal)
    X_train_num_df = X_train[num_cols].copy()
    X_test_num_df = X_test[num_cols].copy()
    final_num_col_names = num_cols.copy() # Daftar akhir kolom numerik

    # --- 1. Handle Datetime Features ---
    logging.info("Processing Datetime features...")
    for col in dt_cols:
        logging.info(f"  -> Engineering features from datetime column '{col}'")
        X_train_dt = pd.to_datetime(X_train[col])
        X_test_dt = pd.to_datetime(X_test[col])
        
        # Ekstrak fitur
        X_train_num_df[f'{col}_hour'] = X_train_dt.dt.hour
        X_train_num_df[f'{col}_dayofweek'] = X_train_dt.dt.dayofweek
        X_train_num_df[f'{col}_month'] = X_train_dt.dt.month
        
        X_test_num_df[f'{col}_hour'] = X_test_dt.dt.hour
        X_test_num_df[f'{col}_dayofweek'] = X_test_dt.dt.dayofweek
        X_test_num_df[f'{col}_month'] = X_test_dt.dt.month
        
        new_dt_features = [f'{col}_hour', f'{col}_dayofweek', f'{col}_month']
        final_num_col_names.extend(new_dt_features)

    # --- 2. Handle Categorical Features (Cardinality Check) ---
    logging.info("Analyzing categorical columns for cardinality...")
    cols_to_one_hot = []
    
    for col in cat_cols:
        unique_count = X_train[col].nunique()
        if unique_count > HIGH_CARDINALITY_THRESHOLD:
            logging.warning(f"  -> WARNING: Dropping high-cardinality feature '{col}'. (Found {unique_count} unique values, threshold is {HIGH_CARDINALITY_THRESHOLD})")
        else:
            cols_to_one_hot.append(col)
            logging.info(f"  -> OK: Keeping low-cardinality feature '{col}' ({unique_count} unique values) for one-hot encoding.")

    # --- 3. Handle Numerical Features (Imputation) ---
    logging.info("Imputing and scaling numerical features...")
    for col in final_num_col_names:
        mean_val = X_train_num_df[col].mean()
        X_train_num_df[col] = X_train_num_df[col].fillna(method='ffill').fillna(method='bfill').fillna(mean_val)
        X_test_num_df[col] = X_test_num_df[col].fillna(method='ffill').fillna(method='bfill').fillna(mean_val)

    # --- 4. Scale Numerical Features ---
    scaler = StandardScaler()
    X_train_num = scaler.fit_transform(X_train_num_df[final_num_col_names])
    X_test_num = scaler.transform(X_test_num_df[final_num_col_names])
    logging.info(f"Numerical features (total {len(final_num_col_names)}) scaled.")

    # --- 5. Handle Categorical Features (One-Hot Encoding) ---
    if cols_to_one_hot:
        X_train_cat = pd.get_dummies(X_train[cols_to_one_hot], drop_first=True, dtype=int)
        X_test_cat = pd.get_dummies(X_test[cols_to_one_hot], drop_first=True, dtype=int)
        
        X_train_cat, X_test_cat = X_train_cat.align(X_test_cat, join='left', axis=1, fill_value=0)
        
        cat_feature_names = X_train_cat.columns.tolist()
        logging.info(f"Categorical features one-hot encoded. Found {len(cat_feature_names)} new features.")
    else:
        X_train_cat = pd.DataFrame(index=X_train.index)
        X_test_cat = pd.DataFrame(index=X_test.index)
        cat_feature_names = []
        logging.info("No low-cardinality categorical features found to encode.")

    # --- 6. Combine Features ---
    all_feature_names = final_num_col_names + cat_feature_names
    
    if not X_train_cat.empty:
        X_train_all = np.concatenate([X_train_num, X_train_cat.values], axis=1)
        X_test_all = np.concatenate([X_test_num, X_test_cat.values], axis=1)
    else:
        X_train_all = X_train_num
        X_test_all = X_test_num

    logging.info(f"Preprocessing complete. Final feature shape: {X_train_all.shape[1]}")
    return X_train_all, X_test_all, scaler, all_feature_names


def encode_labels(y_train, y_test, is_classification):
    # (Tidak berubah)
    if is_classification:
        if y_train.dtype == 'O' or len(np.unique(y_train)) > 2:
            logging.info("Encoding string labels for classification.")
            le = LabelEncoder()
            y_train_enc = le.fit_transform(y_train)
            y_test_enc = le.transform(y_test)
            return y_train_enc, y_test_enc, le
        else:
            logging.info("Labels are already numeric for classification.")
            return np.array(y_train, dtype=int), np.array(y_test, dtype=int), None
    else:
        logging.info("Converting regression target to float.")
        return np.array(y_train, dtype=np.float64), np.array(y_test, dtype=np.float64), None

def clean_nan_from_arrays(X_train, X_test, y_train, y_test):
    # (Tidak berubah)
    logging.info("Cleaning final NaN values from arrays...")
    y_train = np.asarray(y_train, dtype=np.float64).ravel()
    y_test = np.asarray(y_test, dtype=np.float64).ravel()
    X_train_nan_mask = np.isnan(X_train).any(axis=1) if X_train.dtype.kind in 'fc' else np.zeros(len(X_train), dtype=bool)
    y_train_nan_mask = np.isnan(y_train)
    train_valid_mask = ~(X_train_nan_mask | y_train_nan_mask)
    X_train_clean = X_train[train_valid_mask]
    y_train_clean = y_train[train_valid_mask]
    train_dropped = len(X_train) - len(X_train_clean)
    if train_dropped > 0:
        logging.info(f"Dropped {train_dropped} rows with NaN from FINAL training set")
    X_test_nan_mask = np.isnan(X_test).any(axis=1) if X_test.dtype.kind in 'fc' else np.zeros(len(X_test), dtype=bool)
    y_test_nan_mask = np.isnan(y_test)
    test_valid_mask = ~(X_test_nan_mask | y_test_nan_mask)
    X_test_clean = X_test[test_valid_mask]
    y_test_clean = y_test[test_valid_mask]
    test_dropped = len(X_test) - len(X_test_clean)
    if test_dropped > 0:
        logging.info(f"Dropped {test_dropped} rows with NaN from FINAL test set")
    return X_train_clean, X_test_clean, y_train_clean, y_test_clean

def build_model(model_type, input_dim=None, hyperparams=None):
    # (Tidak berubah)
    if hyperparams is None:
        hyperparams = {}
    if model_type == 'rfc':
        logging.info("Building Random Forest Classifier (CPU-only)")
        return RandomForestClassifier(**hyperparams, random_state=42, n_jobs=-1)
    elif model_type == 'rfr':
        logging.info("Building Random Forest Regressor (CPU-only)")
        return RandomForestRegressor(**hyperparams, random_state=42, n_jobs=-1)
    elif 'keras' in model_type:
        raise NotImplementedError("Keras models are now built via 'build_hypermodel' for tuning.")
    else:
        raise ValueError(f"Unsupported model type: {model_type}")

def build_hypermodel(hp, model_type, input_dim, n_classes=2):
    # (Tidak berubah)
    hp_units_1 = hp.Int('units_1', min_value=32, max_value=256, step=32)
    hp_units_2 = hp.Int('units_2', min_value=32, max_value=128, step=32)
    hp_learning_rate = hp.Choice('learning_rate', values=[1e-2, 1e-3, 1e-4])
    model = Sequential([
        Dense(units=hp_units_1, activation='relu', input_shape=(input_dim,)),
        Dense(units=hp_units_2, activation='relu'),
    ])
    if 'clf' in model_type:
        output_activation = 'sigmoid' if n_classes <= 2 else 'softmax'
        output_units = 1 if n_classes <= 2 else n_classes
        loss_function = 'binary_crossentropy' if n_classes <= 2 else 'sparse_categorical_crossentropy'
        metrics = ['accuracy']
        model.add(Dense(output_units, activation=output_activation))
    else:
        model.add(Dense(1, activation='linear'))
        loss_function = 'mse'
        metrics = ['mse', 'mae']
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=hp_learning_rate),
        loss=loss_function,
        metrics=metrics
    )
    return model

def build_default_keras_model(model_type, input_dim, n_classes=2):
    # (Tidak berubah)
    default_hps = kt.HyperParameters()
    default_hps.Int('units_1', default=128)
    default_hps.Int('units_2', default=64)
    default_hps.Choice('learning_rate', default=1e-3)
    logging.info("Building Keras model with default hyperparameters (tuning disabled).")
    return build_hypermodel(default_hps, model_type, input_dim, n_classes)


def save_model(model, path, model_type):
    # (Tidak berubah)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    if 'keras' in model_type:
        if not path.endswith(('.h5', '.keras')):
             path = os.path.join(path, "keras_model")
        model.save(path)
    else:
        if not path.endswith('.joblib'):
            path += '.joblib'
        joblib.dump(model, path)
    logging.info(f"Model saved to: {path}")

# --- Plotting Functions (DIREVISI) ---

# DIREVISI: Memperbaiki tampilan scatter plot untuk regresi
def plot_classification_or_regression(y_true, y_pred, is_classification, plot_filename, confusion_matrix_filename, labels=None):
    plt.figure(figsize=(8, 6))
    if is_classification:
        cm = confusion_matrix(y_true, y_pred)
        sns.heatmap(cm, annot=True, fmt='d', cmap="Blues", cbar=False, 
                    xticklabels=labels if labels is not None else 'auto', 
                    yticklabels=labels if labels is not None else 'auto')
        plt.title("Confusion Matrix")
        plt.ylabel("Actual label")
        plt.xlabel("Predicted label")
        plt.tight_layout()
        plt.savefig(confusion_matrix_filename)
        plt.close()
        logging.info(f"Confusion matrix saved as {confusion_matrix_filename}")
    else:
        # --- REVISI UNTUK REGRESI ---
        # Mengurangi ukuran titik (s=20), mengurangi opacity (alpha=0.3) untuk melihat kepadatan
        plt.scatter(y_true, y_pred, edgecolors='k', c='r', marker='o', alpha=0.3, s=20) 
        plt.title('Predicted vs Actual Values')
        plt.xlabel('Actual')
        plt.ylabel('Predicted')
        
        min_val = min(y_true.min(), y_pred.min())
        max_val = max(y_true.max(), y_pred.max())
        plt.plot([min_val, max_val], [min_val, max_val], 'b--', label='Ideal Prediction')
        plt.legend()
        plt.grid(True, linestyle='--', alpha=0.6) # Tambah grid
        plt.tight_layout()
        plt.savefig(plot_filename)
        plt.close()
        logging.info(f"Regression performance plot saved as {plot_filename}")

# DIREVISI: Memperbaiki format angka pada heatmap
def plot_correlation_heatmap(X, num_cols, filename=DEFAULT_CORR_HEATMAP_FILENAME):
    logging.info(f"Generating correlation heatmap for numerical features...")
    if not num_cols:
        logging.warning("No numerical columns found to plot correlation heatmap.")
        return
    
    # --- REVISI UNTUK UKURAN HEATMAP & FORMAT ANGKA ---
    plt.figure(figsize=(18, 16)) # Ukuran figur lebih besar untuk menampung banyak fitur
    corr_matrix = X[num_cols].corr() 
    
    # Format angka menjadi 2 desimal ('.2f') dan sesuaikan ukuran font annotasi (size=7)
    sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5, annot_kws={"size": 7}) 
    
    plt.title('Feature Correlation Heatmap (Original Numerical Features)', fontsize=16)
    plt.xticks(rotation=45, ha='right', fontsize=10)
    plt.yticks(rotation=0, fontsize=10)
    plt.tight_layout()
    plt.savefig(filename)
    plt.close()
    logging.info(f"Correlation heatmap saved to {filename}")

def plot_keras_history(history, filename_prefix=DEFAULT_KERAS_HISTORY_FILENAME_PREFIX):
    # (Tidak berubah)
    logging.info("Generating Keras training history plots...")
    history_df = pd.DataFrame(history.history)
    plt.figure(figsize=(10, 6))
    loss_plotted = False
    if 'loss' in history_df.columns:
        plt.plot(history_df['loss'], label='Training Loss')
        loss_plotted = True
    if 'val_loss' in history_df.columns:
        plt.plot(history_df['val_loss'], label='Validation Loss')
        loss_plotted = True
    if loss_plotted:
        plt.title('Model Loss Over Epochs')
        plt.ylabel('Loss')
        plt.xlabel('Epoch')
        plt.legend()
        plt.tight_layout()
        plt.savefig(f"{filename_prefix}_loss.png")
        plt.close()
    else:
        plt.close()
    
    metric_keys = ['accuracy', 'acc', 'mse', 'mae']
    train_metric_key = None
    val_metric_key = None
    for key in metric_keys:
        if key in history_df.columns:
            train_metric_key = key
            val_metric_key = f"val_{key}"
            break
    
    metric_plotted = False
    plt.figure(figsize=(10, 6))
    if train_metric_key and train_metric_key in history_df.columns:
        plt.plot(history_df[train_metric_key], label=f'Training {train_metric_key.capitalize()}')
        metric_plotted = True
    if val_metric_key and val_metric_key in history_df.columns:
        plt.plot(history_df[val_metric_key], label=f'Validation {val_metric_key.capitalize()}')
        metric_plotted = True
    if metric_plotted:
        plt.title(f'Model {train_metric_key.capitalize()} Over Epochs')
        plt.ylabel(train_metric_key.capitalize())
        plt.xlabel('Epoch')
        plt.legend()
        plt.tight_layout()
        plt.savefig(f"{filename_prefix}_metric.png")
        plt.close()
    else:
        plt.close()

# DIREVISI: Memperbaiki tampilan scatter plot untuk residual
def plot_regression_diagnostics(y_true, y_pred, filename=DEFAULT_RESIDUAL_PLOT_FILENAME):
    logging.info("Generating regression diagnostic plots...")
    residuals = y_true - y_pred
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7)) 

    # --- REVISI UNTUK SCATTER PLOT RESIDUALS ---
    # Mengurangi ukuran titik (s=20), mengurangi opacity (alpha=0.3)
    sns.scatterplot(x=y_pred, y=residuals, alpha=0.3, ax=ax1, s=20, edgecolor='k', color='r') 
    ax1.axhline(y=0, color='blue', linestyle='--', linewidth=1.5)
    ax1.set_title('Residuals vs. Predicted Values', fontsize=14)
    ax1.set_xlabel('Predicted Values', fontsize=12)
    ax1.set_ylabel('Residuals', fontsize=12)
    ax1.grid(True, linestyle='--', alpha=0.6)

    # Histplot residuals
    sns.histplot(residuals, kde=True, ax=ax2, color='skyblue', edgecolor='black')
    ax2.set_title('Histogram of Residuals', fontsize=14)
    ax2.set_xlabel('Residual Value', fontsize=12)
    ax2.set_ylabel('Frequency', fontsize=12)
    ax2.grid(True, linestyle='--', alpha=0.6)
    
    plt.tight_layout()
    plt.savefig(filename)
    plt.close()
    logging.info(f"Regression diagnostics plot saved to {filename}")

def plot_feature_importance(model, feature_names, filename=DEFAULT_FEATURE_IMPORTANCE_FILENAME):
    # (Tidak berubah)
    logging.info("Generating feature importance plot...")
    if not hasattr(model, 'feature_importances_'):
        logging.warning("Model does not have 'feature_importances_' attribute. Skipping plot.")
        return
    if not feature_names:
        logging.warning("No feature names provided. Skipping feature importance plot.")
        return
    importances = model.feature_importances_
    if len(importances) != len(feature_names):
        logging.warning(f"Feature importance count ({len(importances)}) does not match feature name count ({len(feature_names)}). Skipping plot.")
        return
    importance_df = pd.DataFrame({
        'Feature': feature_names,
        'Importance': importances
    }).sort_values(by='Importance', ascending=False)
    top_features_df = importance_df.head(20)
    plt.figure(figsize=(12, 10))
    sns.barplot(x='Importance', y='Feature', data=top_features_df, palette='viridis')
    plt.title('Top 20 Feature Importances')
    plt.tight_layout()
    plt.savefig(filename)
    plt.close()
    logging.info(f"Feature importance plot saved to {filename}")
# --- End of Plotting Functions ---

def evaluate_model(
    model, X_test, y_test, model_type, 
    classification=True, 
    plot_filenames=None, 
    label_encoder=None,
    feature_names=None
):
    # (Tidak berubah)
    if plot_filenames is None:
        plot_filenames = {}
    logging.info("Predicting on test set...")
    if 'keras' in model_type:
        y_pred = model.predict(X_test)
        if classification:
            if y_pred.ndim > 1 and y_pred.shape[1] > 1:
                y_pred_labels = np.argmax(y_pred, axis=1)
            else:
                y_pred_labels = (y_pred > 0.5).astype(int).reshape(-1)
        else:
            y_pred_labels = y_pred.ravel()
    else:
        y_pred_labels = model.predict(X_test)
    logging.info("Calculating metrics...")
    results = {}
    if classification:
        results['accuracy'] = accuracy_score(y_test, y_pred_labels)
        results['precision'] = precision_score(y_test, y_pred_labels, average='weighted', zero_division=0)
        results['recall'] = recall_score(y_test, y_pred_labels, average='weighted', zero_division=0)
        results['f1'] = f1_score(y_test, y_pred_labels, average='weighted', zero_division=0)
        class_labels = label_encoder.classes_ if label_encoder and hasattr(label_encoder, 'classes_') else None
        class_report = classification_report(
            y_test, y_pred_labels,
            target_names=class_labels,
            zero_division=0
        )
        logging.info("\nClassification Report:\n%s", class_report)
        results['classification_report'] = class_report
        if plot_filenames.get("confusion_matrix"):
            plot_classification_or_regression(
                y_test, y_pred_labels, True, None, 
                plot_filenames["confusion_matrix"], 
                labels=class_labels
            )
        if plot_filenames.get("feature_importance") and feature_names:
            plot_feature_importance(model, feature_names, plot_filenames["feature_importance"])
    else: # Regression
        results['rmse'] = np.sqrt(mean_squared_error(y_test, y_pred_labels))
        results['r2'] = r2_score(y_test, y_pred_labels)
        results['mae'] = mean_absolute_error(y_test, y_pred_labels)
        logging.info(f"RMSE: {results['rmse']:.4f}, MAE: {results['mae']:.4f}, R2: {results['r2']:.4f}")
        if plot_filenames.get("performance_plot"):
            plot_classification_or_regression(
                y_test, y_pred_labels, False, 
                plot_filenames["performance_plot"], None
            )
        if plot_filenames.get("diagnostic_plot"):
            plot_regression_diagnostics(y_test, y_pred_labels, plot_filenames["diagnostic_plot"])
        if plot_filenames.get("feature_importance") and feature_names:
            plot_feature_importance(model, feature_names, plot_filenames["feature_importance"])
    return results

def main(
    data_path=DEFAULT_DATA_PATH,
    model_type=DEFAULT_MODEL_TYPE,
    model_save_path=DEFAULT_MODEL_SAVE_PATH,
    target_column=DEFAULT_TARGET_COLUMN,
    test_ratio=0.2,
    random_seed=42,
    use_gpu=DEFAULT_USE_GPU,
    hyperparams=None,
    plot_performance=True,
    enable_tuning=DEFAULT_ENABLE_TUNING,
    handle_imbalance=DEFAULT_HANDLE_IMBALANCE,
    tuning_iter=DEFAULT_TUNING_ITER
):
    if hyperparams is None:
        hyperparams = {}
        
    configure_gpu(use_gpu=use_gpu)
    set_all_seeds(random_seed)
    
    X, y = load_dataset(data_path, target_column)
    
    # Plot korelasi HANYA pada kolom numerik asli
    num_cols_original, _, _ = identify_feature_types(X)
    if plot_performance:
         plot_correlation_heatmap(X, num_cols_original, DEFAULT_CORR_HEATMAP_FILENAME)

    is_classification = (model_type in ['rfc', 'keras_mlp_clf'])
    
    stratify_data = None
    if is_classification:
        unique_classes, counts = np.unique(y, return_counts=True)
        if len(unique_classes) > 1 and all(counts > 1):
             stratify_data = y
             logging.info("Using stratified split for classification.")
        else:
             logging.warning("Cannot use stratified split (target is uniform or has single-member classes). Using random split.")
             
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_ratio, random_state=random_seed, stratify=stratify_data
    )
    
    logging.info(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")
    
    # Memanggil fungsi preprocessing yang baru
    X_train_pp, X_test_pp, scaler, feature_names = preprocess_features(X_train, X_test)
    
    y_train_enc, y_test_enc, label_encoder = encode_labels(y_train, y_test, is_classification)
    
    X_train_pp, X_test_pp, y_train_enc, y_test_enc = clean_nan_from_arrays(
        X_train_pp, X_test_pp, y_train_enc, y_test_enc
    )
    
    if X_train_pp.shape[0] == 0 or X_test_pp.shape[0] == 0:
        logging.error("FATAL: No data remaining after cleaning NaNs. Exiting.")
        sys.exit(1)
        
    logging.info(f"Final training data shape: {X_train_pp.shape}")
    logging.info(f"Final test data shape: {X_test_pp.shape}")

    input_dim = X_train_pp.shape[1]

    # --- Penanganan Imbalance (Tidak berubah) ---
    class_weight_dict = None
    if is_classification and handle_imbalance:
        logging.info("Handling data imbalance with 'balanced' class weights...")
        try:
            y_train_int = y_train_enc.astype(int)
            unique_classes_int = np.unique(y_train_int)
            weights = compute_class_weight(
                'balanced',
                classes=unique_classes_int,
                y=y_train_int
            )
            class_weight_dict = dict(zip(unique_classes_int, weights))
            logging.info(f"Calculated class weights: {class_weight_dict}")
            if 'keras' not in model_type:
                hyperparams['class_weight'] = class_weight_dict
        except ValueError as e:
            logging.warning(f"Could not compute class weights: {e}")

    # --- Blok Training & Tuning Utama (Tidak berubah) ---
    model = None
    history = None
    
    if enable_tuning:
        logging.info(f"Hyperparameter tuning ENABLED (iterations={tuning_iter})")
        if 'keras' in model_type:
            n_classes = len(np.unique(y_train_enc)) if is_classification else 2
            hypermodel_builder = lambda hp: build_hypermodel(
                hp, model_type=model_type, input_dim=input_dim, n_classes=n_classes
            )
            tuner = kt.RandomizedSearch(
                hypermodel_builder,
                objective='val_accuracy' if is_classification else 'val_mse',
                max_trials=tuning_iter,
                executions_per_trial=1,
                directory='keras_tuner_dir',
                project_name=f'tuning_{model_type}',
                overwrite=True
            )
            early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
            fit_kwargs = {
                'callbacks': [early_stop],
                'class_weight': class_weight_dict if is_classification else None
            }
            logging.info(f"Starting Keras Tuner search...")
            tuner.search(
                X_train_pp, y_train_enc,
                epochs=50,
                validation_data=(X_test_pp, y_test_enc),
                **{k: v for k, v in fit_kwargs.items() if v is not None}
            )
            logging.info("Keras Tuner search complete.")
            tuner.results_summary()
            best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
            model = tuner.hypermodel.build(best_hps)
            logging.info("Retraining best Keras model on full training data...")
            history = model.fit(
                X_train_pp, y_train_enc,
                epochs=100,
                batch_size=hyperparams.get('batch_size', 64),
                validation_data=(X_test_pp, y_test_enc),
                callbacks=[early_stop],
                class_weight=class_weight_dict,
                verbose=2
            )
        else:
            logging.info(f"Starting RandomizedSearchCV for {model_type}...")
            if model_type == 'rfc':
                param_dist = {
                    'n_estimators': randint(100, 500),
                    'max_depth': [None] + list(randint(10, 50).rvs(3)),
                    'min_samples_leaf': randint(1, 5),
                    'min_samples_split': randint(2, 10)
                }
            else: # rfr
                param_dist = {
                    'n_estimators': randint(100, 500),
                    'max_depth': [None] + list(randint(10, 50).rvs(3)),
                    'min_samples_leaf': randint(1, 5),
                    'min_samples_split': randint(2, 10)
                }
            base_model = build_model(model_type, hyperparams=hyperparams) 
            search = RandomizedSearchCV(
                estimator=base_model,
                param_distributions=param_dist,
                n_iter=tuning_iter,
                cv=DEFAULT_CV_FOLDS,
                verbose=2,
                random_state=random_seed,
                n_jobs=-1,
                scoring='f1_weighted' if is_classification else 'r2'
            )
            search.fit(X_train_pp, y_train_enc)
            logging.info(f"RandomizedSearchCV complete. Best CV score ({search.best_score_:.4f})")
            logging.info(f"Best params found: {search.best_params_}")
            model = search.best_estimator_
    else:
        logging.info("Hyperparameter tuning DISABLED. Training single model.")
        if 'keras' in model_type:
            n_classes = len(np.unique(y_train_enc)) if is_classification else 2
            model = build_default_keras_model(model_type, input_dim, n_classes)
            epochs = hyperparams.get('epochs', 50)
            batch_size = hyperparams.get('batch_size', 64)
            early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
            logging.info(f"Training Keras model on: {'/GPU:0' if tf.config.list_physical_devices('GPU') and use_gpu else '/CPU:0'}")
            history = model.fit(
                X_train_pp, y_train_enc,
                epochs=epochs, batch_size=batch_size,
                validation_data=(X_test_pp, y_test_enc),
                callbacks=[early_stop],
                class_weight=class_weight_dict,
                verbose=2
            )
        else:
            model = build_model(model_type, hyperparams=hyperparams)
            model.fit(X_train_pp, y_train_enc)
            
    save_model(model, model_save_path, model_type)
    
    if 'keras' in model_type and history is not None and plot_performance:
        plot_keras_history(history, DEFAULT_KERAS_HISTORY_FILENAME_PREFIX)
    
    logging.info("Evaluating final model on test data...")
    plot_filenames = {}
    if plot_performance:
        if is_classification:
            plot_filenames["confusion_matrix"] = DEFAULT_CONFUSION_MATRIX_FILENAME
            if model_type == 'rfc':
                plot_filenames["feature_importance"] = DEFAULT_FEATURE_IMPORTANCE_FILENAME
        else:
            plot_filenames["performance_plot"] = DEFAULT_PLOT_FILENAME
            plot_filenames["diagnostic_plot"] = DEFAULT_RESIDUAL_PLOT_FILENAME
            if model_type == 'rfr':
                 plot_filenames["feature_importance"] = DEFAULT_FEATURE_IMPORTANCE_FILENAME

    results = evaluate_model(
        model, X_test_pp, y_test_enc,
        model_type, classification=is_classification,
        plot_filenames=plot_filenames,
        label_encoder=label_encoder,
        feature_names=feature_names
    )
    
    # --- Save results (Tidak berubah) ---
    with open("final_evaluation_results.json", "w") as fp:
        class CustomJSONEncoder(json.JSONEncoder):
            def default(self, obj):
                if isinstance(obj, np.integer): return int(obj)
                if isinstance(obj, np.floating): return float(obj)
                if isinstance(obj, np.ndarray): return obj.tolist()
                return super(CustomJSONEncoder, self).default(obj)
        json.dump(results, fp, indent=2, cls=CustomJSONEncoder)
        
    logging.info("==== FINAL EVALUATION RESULTS ====")
    logging.info(f"Results also saved to final_evaluation_results.json")
    for k, v in results.items():
        if k == 'classification_report': continue
        if isinstance(v, float):
            logging.info(f"{k.upper()}: {v:.4f}")
        else:
            logging.info(f"{k.upper()}: {v}")
            
    return results

if __name__ == "__main__":
    # --- Argparse (Tidak berubah) ---
    is_jupyter = False
    try:
        shell = get_ipython().__class__.__name__
        if shell == 'ZMQInteractiveShell': is_jupyter = True
    except NameError:
        is_jupyter = False

    if is_jupyter:
        logging.info("Running in Jupyter/IPython environment with default parameters...")
        # Menekan warning pandas ffill/bfill di Jupyter
        import warnings
        warnings.filterwarnings("ignore", "Series.fillna with 'method' is deprecated")
        main()
    else:
        parser = argparse.ArgumentParser(description="Advanced ML pipeline with HPT, Imbalance Handling, and Cardinality Check.")
        parser.add_argument('--data_path', type=str, default=DEFAULT_DATA_PATH, help="Path to the input CSV file.")
        parser.add_argument('--target_column', type=str, default=DEFAULT_TARGET_COLUMN, help="Name of the target variable column.")
        parser.add_argument('--model_type', type=str, default=DEFAULT_MODEL_TYPE, 
                            choices=['rfc', 'rfr', 'keras_mlp_clf', 'keras_mlp_reg'],
                            help="Type of model to train.")
        parser.add_argument('--model_save_path', type=str, default=DEFAULT_MODEL_SAVE_PATH, help="Path to save the trained model.")
        parser.add_argument('--test_ratio', type=float, default=0.2, help="Proportion of data to use for the test set.")
        parser.add_argument('--random_seed', type=int, default=42, help="Random seed for reproducibility.")
        parser.add_argument('--no_gpu', action='store_true', help="Disable GPU usage even if available.")
        parser.add_argument('--no_plots', action='store_true', help="Disable generation of performance plots.")
        parser.add_argument('--no_tuning', action='store_true', help="Disable hyperparameter tuning.")
        parser.add_argument('--no_imbalance', action='store_true', help="Disable automatic class weight handling.")
        parser.add_argument('--tuning_iter', type=int, default=DEFAULT_TUNING_ITER, help="Number of iterations for RandomizedSearch (HPT).")

        args = parser.parse_args()
        
        main(
            data_path=args.data_path,
            model_type=args.model_type,
            model_save_path=args.model_save_path,
            target_column=args.target_column,
            test_ratio=args.test_ratio,
            random_seed=args.random_seed,
            use_gpu=not args.no_gpu,
            plot_performance=not args.no_plots,
            enable_tuning=not args.no_tuning,
            handle_imbalance=not args.no_imbalance,
            tuning_iter=args.tuning_iter
        )

2025-11-15 17:56:28,273 [INFO] Running in Jupyter/IPython environment with default parameters...
2025-11-15 17:56:28,275 [INFO] ==================================================
2025-11-15 17:56:28,276 [INFO] GPU CONFIGURATION
2025-11-15 17:56:28,278 [INFO] ✓ Found 1 GPU(s):
2025-11-15 17:56:28,279 [INFO]   - GPU 0: /physical_device:GPU:0
2025-11-15 17:56:28,280 [WARNING] ✗ Tidak dapat mengatur memory growth (mungkin runtime sudah diinisialisasi?): Cannot set memory growth on device when virtual devices configured
2025-11-15 17:56:28,281 [INFO] TensorFlow version: 2.20.0
2025-11-15 17:56:28,282 [INFO] GPU Available: True
2025-11-15 17:56:28,283 [INFO] ==================================================
2025-11-15 17:56:28,285 [INFO] Loading dataset from: FINAL_Merged_SmartMeter_dan_Cuaca.csv
2025-11-15 17:56:28,368 [INFO] Dropped 685 rows with NaN in target column 'Konsumsi Energi'
2025-11-15 17:56:28,371 [INFO] Loaded dataset with shape (9616, 47)
2025-11-15 17:56:28,378 [INFO]   -> K

Fitting 3 folds for each of 10 candidates, totalling 30 fits
[CV] END max_depth=38, min_samples_leaf=4, min_samples_split=6, n_estimators=120; total time=  13.3s
[CV] END max_depth=38, min_samples_leaf=4, min_samples_split=6, n_estimators=120; total time=  13.0s
[CV] END max_depth=38, min_samples_leaf=4, min_samples_split=6, n_estimators=370; total time=  39.3s
[CV] END max_depth=38, min_samples_leaf=4, min_samples_split=6, n_estimators=370; total time=  39.3s
[CV] END max_depth=38, min_samples_leaf=4, min_samples_split=6, n_estimators=120; total time=  12.8s
[CV] END max_depth=38, min_samples_leaf=4, min_samples_split=6, n_estimators=370; total time=  39.6s
[CV] END max_depth=38, min_samples_leaf=2, min_samples_split=4, n_estimators=314; total time=  38.9s
[CV] END max_depth=38, min_samples_leaf=2, min_samples_split=4, n_estimators=314; total time=  38.9s
[CV] END max_depth=38, min_samples_leaf=2, min_samples_split=4, n_estimators=314; total time=  39.5s
[CV] END max_depth=38, min_sam

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import io

# Mengatur style visualisasi agar lebih menarik
sns.set_theme(style="whitegrid")

# --- Muat Data Anda ---
# SAYA MENGGUNAKAN DATA SAMPEL 25 BARIS DARI ANDA
# Ganti bagian ini dengan file Anda yang berisi 500 baris.

# Data sampel Anda sebagai string (pastikan pemisahnya adalah Tab '\t')
data_sample = """
id_id	id_n_id	id_time	id_meter_id	id_stand_energy_kirim	id_stand_energy_terima	id_v1	id_v2	id_v3	id_i1	id_i2	id_i3	id_neutral_current	id_daya_aktif_plus	id_daya_aktif_minus	id_frequency	id_power_factor	id_meter_status	id_stand_energi_kirim_last_month	id_stand_energi_terima_last_month	id_create_date	updated_at	created_at	reactive_energy_import	reactive_energy_export	apparent_energy_import	apparent_energy_export	rssi	lokasi
1	1	2023-09-22 14:00:03.000	251400184	289111.0	0	209	209	209	10	10	10				50.03	0.9896999999999999				2023-09-22 14:00:22.000	2023-09-22 14:00:22.000	2023-09-22 14:00:22.000	855	2866	292070	0		61
2	1	2023-09-22 14:05:03.000	251400184	294268.0	0	209	209	209	10	10	10				49.980.000.000.000.000	0.9906999999999999				2023-09-22 14:05:29.000	2023-09-22 14:05:29.000	2023-09-22 14:05:29.000	855	2989	297295	0		61
3	4	2023-09-22 14:05:03.000	251400179	308756.0	0	209	209	209	10	10	10				49.95	0.9904999999999999				2023-09-22 14:05:35.000	2023-09-22 14:05:35.000	2023-09-22 14:05:35.000	47	10137	312092	0		undefined
4	1	2023-09-22 14:10:03.000	251400184	299527.0	0	208	208	208	11	11	11				50	0.9914999999999999				2023-09-22 14:10:23.000	2023-09-22 14:10:23.000	2023-09-22 14:10:23.000	855	3107	302606	0		61
5	8	2023-09-22 14:10:03.000	251400177	306579.0	0	208	208	208	11	11	11				50	0.9906999999999999				2023-09-22 14:10:24.000	2023-09-22 14:10:24.000	2023-09-22 14:10:24.000	55	8783	309851	0		undefined
6	2	2023-09-22 14:10:03.000	251400172	295552.0	0	208	208	208	9	11	11				50.01	0.9803				2023-09-22 14:10:26.000	2023-09-22 14:10:26.000	2023-09-22 14:10:26.000	20	30384	301388	0		undefined
7	9	2023-09-22 14:10:03.000	251400182	302485.0	0	208	208	208	11	11	11				49.99	0.9904				2023-09-22 14:10:29.000	2023-09-22 14:10:29.000	2023-09-22 14:10:29.000	571	3966	305604	0		undefined
8	5	2023-09-22 14:10:03.000	251400178	312697.0	0	208	208	208	11	11	11				49.99	0.9901999999999999				2023-09-22 14:10:31.000	2023-09-22 14:10:31.000	2023-09-22 14:10:31.000	48	9986	316061	0		undefined
9	6	2023-09-22 14:10:03.000	251400181	311380.0	0	208	208	208	11	11	11				50	0.9892				2023-09-22 14:10:32.000	2023-09-22 14:10:32.000	2023-09-22 14:10:32.000	46	9996	314741	0		undefined
10	2	2023-09-22 14:15:03.000	251400172	300580.0	0	209	209	209	9	10	10				49.99	0.9806999999999999				2023-09-22 14:15:24.000	2023-09-22 14:15:24.000	2023-09-22 14:15:24.000	20	30977	306516	0		undefined
11	1	2023-09-22 14:15:03.000	251400184	304780.0	0	209	209	209	10	10	10				49.99	0.9908999999999999				2023-09-22 14:15:27.000	2023-09-22 14:15:27.000	2023-09-22 14:15:27.000	855	3230	307907	0		61
12	4	2023-09-22 14:15:03.000	251400179	319340.0	0	209	209	209	10	10	10				50	0.9892				2023-09-22 14:15:29.000	2023-09-22 14:15:29.000	2023-09-22 14:15:29.000	47	10647	322786	0		undefined
13	5	2023-09-22 14:15:03.000	251400178	317984.0	0	209	209	209	10	10	10				49.97	0.9897999999999999				2023-09-22 14:15:34.000	2023-09-22 14:15:34.000	2023-09-22 14:15:34.000	48	10238	321401	0		undefined
14	6	2023-09-22 14:15:03.000	251400181	316651.0	0	209	209	209	10	10	10				50	0.9893				2023-09-22 14:15:45.000	2023-09-22 14:15:45.000	2023-09-22 14:15:45.000	46	10248	320065	0		undefined
15	5	2023-09-22 14:20:03.000	251400178	323285.0	0	208	208	208	10	10	10				50	0.9903				2023-09-22 14:20:29.000	2023-09-22 14:20:29.000	2023-09-22 14:20:29.000	48	10496	326756	0		undefined
16	8	2023-09-22 14:20:03.000	251400177	317174.0	0	208	208	208	10	10	10				49.99	0.9901999999999999				2023-09-22 14:20:29.000	2023-09-22 14:20:29.000	2023-09-22 14:20:29.000	55	9258	320552	0		undefined
17	9	2023-09-22 14:20:03.000	251400182	313011.0	0	208	208	208	10	10	10				49.980.000.000.000.000	0.9899999999999999				2023-09-22 14:20:31.000	2023-09-22 14:20:31.000	2023-09-22 14:20:31.000	571	4258	316229	0		undefined
18	1	2023-09-22 14:20:03.000	251400184	310046.0	0	208	208	208	10	10	10				50.01	0.9906999999999999				2023-09-22 14:20:32.000	2023-09-22 14:20:32.000	2023-09-22 14:20:32.000	855	3359	313222	0		61
19	2	2023-09-22 14:20:03.000	251400172	305620.0	0	208	208	208	9	10	10				49.980.000.000.000.000	0.9798999999999999				2023-09-22 14:20:40.000	2023-09-22 14:20:40.000	2023-09-22 14:20:40.000	20	31575	311659	0		undefined
20	1	2023-09-22 14:25:03.000	251400184	315345.0	0	209	209	209	10	10	10				49.99	0.9899999999999999				2023-09-22 14:25:29.000	2023-09-22 14:25:29.000	2023-09-22 14:25:29.000	855	3481	318571	0		61
21	2	2023-09-22 14:25:03.000	251400172	310692.0	0	209	209	209	9	10	10				49.99	0.9803				2023-09-22 14:25:29.000	2023-09-22 14:25:29.000	2023-09-22 14:25:29.000	20	32171	316834	0		undefined
22	4	2023-09-22 14:25:03.000	251400179	329976.0	0	209	209	209	10	10	10				50.03	0.9890999999999999				2023-09-22 14:25:31.000	2023-09-22 14:25:31.000	2023-09-22 14:25:31.000	47	11169	333532	0		undefined
23	5	2023-09-22 14:25:03.000	251400178	328618.0	0	209	209	209	10	10	10				50	0.9897999999999999				2023-09-22 14:25:32.000	2023-09-22 14:25:32.000	2023-09-22 14:25:32.000	48	10748	332144	0		undefined
24	6	2023-09-22 14:25:03.000	251400181	327255.0	0	209	209	209	10	10	10				50.02	0.9889999999999999				2023-09-22 14:25:41.000	2023-09-22 14:25:41.000	2023-09-22 14:25:41.000	46	10757	330778	0		undefined
25	5	2023-09-22 14:30:03.000	251400178	333846.0	0	209	209	209	10	10	10				50.04	0.9898999999999999				2023-09-22 14:30:24.000	2023-09-22 14:30:24.000	2023-09-22 14:30:24.000	48	10998	337425	0		undefined
"""

# !! GANTI BARIS INI UNTUK MEMBACA FILE ANDA !!
# df = pd.read_csv('nama_file_500_data.csv', sep='\t')
df = pd.read_csv('datasmartmeter_sept_2023_januari_2025.csv', sep='\t')

print(f"Data awal dimuat: {df.shape[0]} baris, {df.shape[1]} kolom")
df.head()

Data awal dimuat: 3673260 baris, 1 kolom


,"id_id,""id_n_id"",""id_time"",""id_meter_id"",""id_stand_energy_kirim"",""id_stand_energy_terima"",""id_v1"",""id_v2"",""id_v3"",""id_i1"",""id_i2"",""id_i3"",""id_neutral_current"",""id_daya_aktif_plus"",""id_daya_aktif_minus"",""id_frequency"",""id_power_factor"",""id_meter_status"",""id_stand_energi_kirim_last_month"",""id_stand_energi_terima_last_month"",""id_create_date"",""updated_at"",""created_at"",""reactive_energy_import"",""reactive_energy_export"",""apparent_energy_import"",""apparent_energy_export"",""rssi"",""lokasi"""
0,"1,1,2023-09-22 14:00:03.000,""251400184"",289111..."
1,"2,1,2023-09-22 14:05:03.000,""251400184"",294268..."
2,"3,4,2023-09-22 14:05:03.000,""251400179"",308756..."
3,"4,1,2023-09-22 14:10:03.000,""251400184"",299527..."
4,"5,8,2023-09-22 14:10:03.000,""251400177"",306579..."
